In [ ]:
import numpy as np
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import json

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


class MetageneEmbeddingExtractor:
    """
    Класс для загрузки METAGENE-1 и извлечения эмбеддингов из списка ДНК/РНК последовательностей.
    Эмбеддинг каждой последовательности — это mean pooling по последнему скрытому слою.
    """
    def __init__(self,
                 model_name: str = "metagene-ai/METAGENE-1",
                 device: str = None,
                 torch_dtype=torch.bfloat16,
                 batch_size: int = 8):
        self.model_name = model_name
        self.batch_size = batch_size

        # 1) определяем устройство
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device)

        # 2) загружаем токенизатор и модель
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name, trust_remote_code=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            device_map="auto",
            trust_remote_code=True,
            output_hidden_states=True,
            return_dict=True
        )
        self.model.eval()

    def extract_embeddings(self, sequences, batch_size=None):
        """
        Принимает список строк (ДНК/РНК последовательностей), 
        возвращает numpy.ndarray размера (len(sequences), hidden_size).
        """
        if batch_size is None:
            batch_size = self.batch_size

        all_embs = []
        with torch.no_grad():
            for i in range(0, len(sequences), batch_size):
                batch_seqs = sequences[i:i+batch_size]
                inputs = self.tokenizer(
                    batch_seqs,
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                ).to(self.device)

                out = self.model(**inputs)
                last_hidden = out.hidden_states[-1]  # (B, L, H)
                embs = last_hidden.mean(dim=1)       # (B, H)

                # Здесь приводим к float32, чтобы .numpy() сработал
                embs = embs.to(torch.float32)

                all_embs.append(embs.cpu().numpy())

        return np.vstack(all_embs)



ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")
train_ds, test_ds = ds['train'], ds['test']

extractor = MetageneEmbeddingExtractor()

PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}
PATH_TO_SAVE_OUTPUTS = '.'
BATCH_SIZE = 12

# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task']==t)
    te = test_ds.filter(lambda x, t=task: x['task']==t)
    seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
    seqs_te, y_te = te['sequence'], np.array(te['label'])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**PARAMS_LOGREG)
    Xf = X_tr.reshape(-1,1) if X_tr.ndim==1 or X_tr.shape[1]==1 else X_tr
    Xt = X_te.reshape(-1,1) if X_te.ndim==1 or X_te.shape[1]==1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    with open(f'{PATH_TO_SAVE_OUTPUTS}/results_metagene_task-{task}_baseline.json','w') as f:
        json.dump(baseline, f, indent=4)

# Few-shot эксперименты
def few_shot(train, test, ks=(1,5,10,20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task']==t)
        te = test.filter(lambda x, t=task: x['task']==t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        res[task] = {}
        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())
                X_k = extractor.extract_embeddings(
                    [seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE
                )
                y_k = y_tr[idxs]
                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1,1) if X_k.ndim==1 or X_k.shape[1]==1 else X_k
                Xt = X_te.reshape(-1,1) if X_te.ndim==1 or X_te.shape[1]==1 else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average='macro'))
            res[task][k] = {
                'accuracy': float(np.mean(accs)),
                'f1_score': float(np.mean(f1s))
            }
            with open(f'{PATH_TO_SAVE_OUTPUTS}/results_metagene_task-{task}_k-{k}.json','w') as f:
                json.dump(res, f, indent=4)
    return res

results_kshot = few_shot(train_ds, test_ds)


output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
with open(f'{PATH_TO_SAVE_OUTPUTS}/results_metagene.json','w') as f:
    json.dump(output, f, indent=4)


/home/mikhail_nuridinov/notebooks/models/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-19 21:34:14.819476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752950054.842018 3383090 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752950054.848932 3383090 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752950054.867255 3383090 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same targ